# LeetCode 2656 — Maximum Sum With Exactly K Elements bằng Python

Link đề: https://leetcode.com/problems/maximum-sum-with-exactly-k-elements/ · Độ khó: **Easy** · Chủ đề: Array, Greedy

Skeleton LeetCode:

```python
class Solution:
    def maximizeSum(self, nums: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **tham lam (greedy)**: thực hiện đúng `k` lần thao tác "chọn một phần tử `m`, xoá nó, thêm `m + 1` vào mảng, cộng `m` vào điểm" để điểm cuối cùng **lớn nhất**.
2. Biết cách **mô phỏng** đề bài bằng danh sách (`max`, `remove`, `append`) và bằng **heap** (`heapq`, hàng đợi ưu tiên) — kể cả mẹo dùng số âm để biến min-heap thành max-heap.
3. Nhận ra **quy luật toán học**: luôn chọn phần tử lớn nhất `M`, lần sau phần tử lớn nhất chính là `M + 1`, rồi `M + 2`, ... nên điểm là `M + (M+1) + ... + (M+k-1) = k·M + k(k-1)/2`.
4. Viết 5 cách: mô phỏng bằng danh sách, mô phỏng bằng heap, vòng lặp với một biến, công thức cấp số cộng, và `sum(range(...))`; đối chiếu với **vét cạn mọi cách chọn** trên dữ liệu nhỏ để chứng minh tham lam đúng.
5. Tránh các lỗi: chọn phần tử nhỏ nhất, chỉ cộng `k` lần giá trị `M`, sai cận của tổng (`k(k+1)/2` thay vì `k(k-1)/2`), và sửa mảng trong lúc duyệt.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` (chỉ số từ 0) và số nguyên `k`. Bạn phải thực hiện thao tác sau **đúng `k` lần** để tối đa hoá điểm:

1. Chọn một phần tử `m` trong `nums`.
2. **Xoá** phần tử `m` khỏi mảng.
3. **Thêm** một phần tử mới có giá trị `m + 1` vào mảng.
4. **Cộng** `m` vào điểm.

Trả về **điểm lớn nhất** có thể đạt được sau đúng `k` lần thao tác.

| Ví dụ | nums, k | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,2,3,4,5]`, `3` | `18` | Lần 1 chọn `5` (mảng thành `[1,2,3,4,6]`), lần 2 chọn `6`, lần 3 chọn `7` ⇒ `5 + 6 + 7 = 18`. |
| 2 | `[5,5,5]`, `2` | `11` | Lần 1 chọn `5` (mảng `[5,5,6]`), lần 2 chọn `6` ⇒ `5 + 6 = 11`. |

Constraints:

- `1 <= nums.length <= 100`
- `1 <= nums[i] <= 100`
- `1 <= k <= 100`

In [1]:
nums, k = [1, 2, 3, 4, 5], 3
score = 0
for step in range(1, k + 1):
    m = max(nums)               # thử chọn phần tử lớn nhất
    nums.remove(m)
    nums.append(m + 1)
    score += m
    print(f"lần {step}: chọn m={m}, mảng mới={nums}, điểm={score}")

lần 1: chọn m=5, mảng mới=[1, 2, 3, 4, 6], điểm=5
lần 2: chọn m=6, mảng mới=[1, 2, 3, 4, 7], điểm=11
lần 3: chọn m=7, mảng mới=[1, 2, 3, 4, 8], điểm=18


## 3. Kiến thức nền

### 3.1. `max`, `remove`, `append` trên danh sách

- `max(nums)` trả phần tử lớn nhất.
- `nums.remove(x)` xoá **một** phần tử đầu tiên bằng `x` (không xoá hết các số trùng); nếu không có `x` thì báo `ValueError`.
- `nums.append(y)` thêm `y` vào cuối danh sách.

Ba thao tác đó chính xác tương ứng bước 1–3 của đề.

In [2]:
nums = [5, 5, 5]
m = max(nums)
nums.remove(m)        # chỉ xoá MỘT số 5
print(nums)           # [5, 5]
nums.append(m + 1)
print(nums)           # [5, 5, 6]

[5, 5]
[5, 5, 6]


### 3.2. Heap (hàng đợi ưu tiên) với `heapq`

`heapq` của Python cài đặt **min-heap**: phần tử **nhỏ nhất** luôn ở vị trí `0` và lấy ra nhanh. Các hàm chính:

- `heapq.heapify(lst)`: biến danh sách thành heap tại chỗ, `O(n)`.
- `heapq.heappop(h)`: lấy và xoá phần tử nhỏ nhất, `O(log n)`.
- `heapq.heappush(h, x)`: thêm `x`, `O(log n)`.
- `heapq.heapreplace(h, x)`: lấy phần tử nhỏ nhất rồi thêm `x` (nhanh hơn pop rồi push).

**Mẹo max-heap:** lưu số **âm** `-x`. Phần tử nhỏ nhất của `-x` ứng với phần tử **lớn nhất** của `x`.

In [3]:
import heapq
h = [3, 1, 4, 1, 5]
heapq.heapify(h)
print(heapq.heappop(h))        # 1 (nhỏ nhất)
heapq.heappush(h, 0)
print(heapq.heappop(h))        # 0

# max-heap bằng số âm
nums = [3, 1, 4, 1, 5]
mh = [-x for x in nums]
heapq.heapify(mh)
print(-heapq.heappop(mh))      # 5 (lớn nhất)

1
0
5


### 3.3. Tổng một dãy liên tiếp (cấp số cộng)

Tổng `M + (M + 1) + ... + (M + k - 1)` gồm `k` số hạng, mỗi số là `M` cộng thêm `0, 1, ..., k - 1`:

`= k·M + (0 + 1 + ... + (k - 1)) = k·M + k(k - 1)/2`

Ví dụ `M = 5`, `k = 3`: `5 + 6 + 7 = 18`; công thức `3·5 + 3·2/2 = 15 + 3 = 18`.

In [4]:
M, k = 5, 3
print(5 + 6 + 7, k * M + k * (k - 1) // 2)    # 18 18
print(sum(range(M, M + k)))                    # 18: range(5, 8) = 5, 6, 7

18 18
18


### 3.4. Tham lam (greedy) là gì?

**Tham lam** nghĩa là ở mỗi bước chọn lựa chọn tốt nhất **ngay lúc này** và hy vọng nó dẫn tới kết quả tốt nhất toàn cục. Không phải bài nào tham lam cũng đúng; ta cần **chứng minh**. Ở bài này, chứng minh gọn nằm ở mục 10: sau mỗi thao tác, phần tử lớn nhất của mảng tăng **tối đa 1**, nên điểm thao tác thứ `t` không thể vượt `M + t`.

In [5]:
# thử so sánh tham lam với chọn phần tử nhỏ nhất
def simulate(nums, k, chon):
    nums = nums[:]
    score = 0
    for _ in range(k):
        m = chon(nums)
        nums.remove(m)
        nums.append(m + 1)
        score += m
    return score

print("chọn lớn nhất:", simulate([1, 2, 3, 4, 5], 3, max))    # 18
print("chọn nhỏ nhất:", simulate([1, 2, 3, 4, 5], 3, min))    # 1 + 2 + 3 = 6

chọn lớn nhất: 18
chọn nhỏ nhất: 5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mỗi lần ta "lấy điểm" bằng giá trị phần tử được chọn. Muốn điểm lớn ⇒ chọn phần tử **lớn**. Điều thú vị: sau khi chọn `m`, phần tử `m + 1` được **thêm** vào mảng, và nó lớn hơn `m` nên (nếu `m` là lớn nhất) nó **trở thành phần tử lớn nhất mới**. Do đó tiếp tục chọn nó.

Tính tay `nums = [1, 2, 3, 4, 5]`, `k = 3`:

1. Lớn nhất `5` ⇒ lấy `5`, thêm `6` ⇒ mảng `[1,2,3,4,6]`.
2. Lớn nhất `6` ⇒ lấy `6`, thêm `7` ⇒ `[1,2,3,4,7]`.
3. Lớn nhất `7` ⇒ lấy `7`, thêm `8`.

Điểm `5 + 6 + 7 = 18`. Ta thấy các giá trị lấy ra là `5, 6, 7` — **các phần tử còn lại `1, 2, 3, 4` không bao giờ được dùng**.

Kết luận trực giác: **chỉ cần biết phần tử lớn nhất `M`**, đáp án là `M + (M + 1) + ... + (M + k − 1)`.

In [6]:
nums = [3, 9, 2]
M = max(nums)
k = 4
print("các giá trị lấy ra:", [M + i for i in range(k)], "-> tổng", sum(M + i for i in range(k)))

các giá trị lấy ra: [9, 10, 11, 12] -> tổng 42


## 5. Cách 1 — Brute Force / baseline: mô phỏng bằng danh sách

**Ý tưởng:** làm **đúng như đề**: lặp `k` lần; mỗi lần tìm `m = max(nums)`, xoá `m`, thêm `m + 1`, cộng `m` vào điểm. Chọn phần tử lớn nhất là lựa chọn tham lam.

Ta làm việc trên **bản sao** `a = nums[:]` để không làm hỏng dữ liệu gốc (an toàn khi dùng lại).

In [7]:
from typing import List

class SolutionSimulate:
    def maximizeSum(self, nums: List[int], k: int) -> int:
        a = nums[:]                # bản sao
        score = 0
        for _ in range(k):
            m = max(a)             # O(n)
            a.remove(m)            # O(n)
            a.append(m + 1)        # O(1)
            score += m
        return score

In [8]:
sol = SolutionSimulate()
print(sol.maximizeSum([1, 2, 3, 4, 5], 3))    # 18
print(sol.maximizeSum([5, 5, 5], 2))          # 11
print(sol.maximizeSum([7], 1))                # 7

18
11
7


**Độ phức tạp:** mỗi lần `max` và `remove` đều `O(n)` nên Time `O(k · n)` (tối đa `100 × 100`); Space `O(n)` cho bản sao.

**Nút thắt:** mỗi lần ta quét lại cả mảng để tìm lớn nhất, mặc dù ta biết **phần tử lớn nhất kế tiếp** chính là phần tử vừa thêm vào. Có thể không cần quét.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — dùng heap.** Cần "lấy phần tử lớn nhất, rồi chèn phần tử mới" lặp lại ⇒ đúng bài của hàng đợi ưu tiên (max-heap): mỗi thao tác `O(log n)` thay vì `O(n)`.

**Đột phá 2 — chỉ cần một biến.** Sau khi lấy `m` (lớn nhất) và thêm `m + 1`, phần tử `m + 1` **lớn hơn mọi phần tử còn lại** (vì `m` đã là lớn nhất). Vậy lần sau lấy chính nó, rồi `m + 2`, ... ⇒ ta không cần giữ cả mảng, chỉ cần biến `cur` bắt đầu bằng `M = max(nums)`, mỗi bước `score += cur; cur += 1`.

**Đột phá 3 — công thức.** Cộng `k` số hạng `M, M+1, ..., M+k−1`: `k·M + k(k−1)/2`. Thuật toán chỉ còn việc tìm `max(nums)`.

In [9]:
M, k = 5, 3
cur, score = M, 0
for _ in range(k):
    score += cur
    cur += 1
print(score, "== công thức:", k * M + k * (k - 1) // 2)

18 == công thức: 18


## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng bằng max-heap

Đẩy `-x` của mọi phần tử vào heap. Mỗi lần: lấy phần tử nhỏ nhất của heap (tức `-m` với `m` lớn nhất), `score += m`, rồi đẩy `-(m + 1)` vào. `heapq.heapreplace` làm cả hai việc.

In [10]:
import heapq

class SolutionHeap:
    def maximizeSum(self, nums: List[int], k: int) -> int:
        heap = [-x for x in nums]      # max-heap bằng số âm
        heapq.heapify(heap)
        score = 0
        for _ in range(k):
            m = -heap[0]               # phần tử lớn nhất (xem, chưa xoá)
            score += m
            heapq.heapreplace(heap, -(m + 1))   # xoá m, thêm m + 1
        return score

print(SolutionHeap().maximizeSum([1, 2, 3, 4, 5], 3))    # 18
print(SolutionHeap().maximizeSum([5, 5, 5], 2))          # 11

18
11


**Độ phức tạp:** `heapify` `O(n)`, mỗi bước `O(log n)` ⇒ Time `O(n + k log n)`; Space `O(n)`.

### Cách 3 — Một biến `cur`, không giữ mảng

Chỉ cần `cur = max(nums)`; lặp `k` lần: cộng `cur` rồi tăng `cur` thêm `1`.

In [11]:
class SolutionLoop:
    def maximizeSum(self, nums: List[int], k: int) -> int:
        cur = max(nums)
        score = 0
        for _ in range(k):
            score += cur
            cur += 1
        return score

print(SolutionLoop().maximizeSum([1, 2, 3, 4, 5], 3))    # 18
print(SolutionLoop().maximizeSum([5, 5, 5], 2))          # 11

18
11


**Độ phức tạp:** Time `O(n + k)` (tìm max rồi `k` vòng); Space `O(1)`.

### Cách 4 — Công thức cấp số cộng `k·M + k(k−1)/2`

Thay vòng lặp bằng công thức: `M = max(nums)`, trả `k * M + k * (k - 1) // 2`. Dùng `//` vì `k(k − 1)` luôn chẵn.

In [12]:
class SolutionFormula:
    def maximizeSum(self, nums: List[int], k: int) -> int:
        M = max(nums)
        return k * M + k * (k - 1) // 2

print(SolutionFormula().maximizeSum([1, 2, 3, 4, 5], 3))    # 18
print(SolutionFormula().maximizeSum([5, 5, 5], 2))          # 11

18
11


**Độ phức tạp:** Time `O(n)` (chỉ để tìm `max`); Space `O(1)`.

### Cách 5 — `sum(range(M, M + k))`

Dãy `M, M+1, ..., M+k−1` chính là `range(M, M + k)`; `sum` cộng chúng lại. Là cách viết **dễ đọc nhất**: nhìn vào code là thấy ngay "tổng k số liên tiếp bắt đầu từ M".

In [13]:
class SolutionRangeSum:
    def maximizeSum(self, nums: List[int], k: int) -> int:
        M = max(nums)
        return sum(range(M, M + k))

print(SolutionRangeSum().maximizeSum([1, 2, 3, 4, 5], 3))    # 18
print(SolutionRangeSum().maximizeSum([5, 5, 5], 2))          # 11

18
11


**Độ phức tạp:** `max` `O(n)` và `sum(range(...))` `O(k)` ⇒ Time `O(n + k)`; Space `O(1)` (`range` là đối tượng lười, không tạo danh sách).

## 8. TRACE từng bước

Trace Cách 1 trên `nums = [1,2,3,4,5]`, `k = 3`:

| lần | mảng trước | `m = max` | xoá `m`, thêm `m+1` ⇒ mảng sau | điểm |
|---|---|---|---|---|
| 1 | `[1,2,3,4,5]` | 5 | `[1,2,3,4,6]` | 5 |
| 2 | `[1,2,3,4,6]` | 6 | `[1,2,3,4,7]` | 11 |
| 3 | `[1,2,3,4,7]` | 7 | `[1,2,3,4,8]` | 18 |

Trace Cách 3 (một biến) cho thấy `cur` đi qua đúng `5, 6, 7`.

In [14]:
def trace_simulate(nums, k):
    a = nums[:]
    score = 0
    for step in range(1, k + 1):
        m = max(a)
        before = a[:]
        a.remove(m)
        a.append(m + 1)
        score += m
        print(f"lần {step}: {before} -> chọn m={m} -> {a}, điểm={score}")
    return score

trace_simulate([1, 2, 3, 4, 5], 3)
trace_simulate([5, 5, 5], 2)

lần 1: [1, 2, 3, 4, 5] -> chọn m=5 -> [1, 2, 3, 4, 6], điểm=5
lần 2: [1, 2, 3, 4, 6] -> chọn m=6 -> [1, 2, 3, 4, 7], điểm=11
lần 3: [1, 2, 3, 4, 7] -> chọn m=7 -> [1, 2, 3, 4, 8], điểm=18
lần 1: [5, 5, 5] -> chọn m=5 -> [5, 5, 6], điểm=5
lần 2: [5, 5, 6] -> chọn m=6 -> [5, 5, 7], điểm=11


11

In [15]:
def trace_cur(nums, k):
    cur = max(nums)
    score = 0
    print(f"M = {cur}")
    for step in range(1, k + 1):
        score += cur
        print(f"lần {step}: cộng cur={cur} -> điểm={score}")
        cur += 1
    print("kết quả:", score, "| công thức:", k * max(nums) + k * (k - 1) // 2)
    return score

trace_cur([1, 2, 3, 4, 5], 3)

M = 5
lần 1: cộng cur=5 -> điểm=5
lần 2: cộng cur=6 -> điểm=11
lần 3: cộng cur=7 -> điểm=18
kết quả: 18 | công thức: 18


18

## 9. Code nộp LeetCode

Dùng Cách 4: tìm `max` rồi dùng công thức. Ngắn, `O(n)`, không vòng lặp phụ.

In [16]:
from typing import List

class Solution:
    def maximizeSum(self, nums: List[int], k: int) -> int:
        M = max(nums)
        return k * M + k * (k - 1) // 2

In [17]:
print(Solution().maximizeSum([1, 2, 3, 4, 5], 3))    # 18
print(Solution().maximizeSum([5, 5, 5], 2))          # 11

18
11


## 10. Vì sao thuật toán đúng

Gọi `M = max(nums)`.

**Cận trên.** Sau `t` thao tác (`t = 0, 1, ...`), phần tử lớn nhất của mảng không vượt quá `M + t`. Chứng minh quy nạp: ban đầu lớn nhất là `M`. Mỗi thao tác chọn một phần tử `m <= (lớn nhất hiện tại)` rồi thêm `m + 1 <= (lớn nhất hiện tại) + 1`. Các phần tử còn lại không tăng, nên lớn nhất mới `<= (lớn nhất cũ) + 1`.

Vậy điểm ở thao tác thứ `t + 1` (đánh số từ `0`) bằng giá trị phần tử được chọn `<= M + t`. Tổng `k` thao tác:

`điểm <= M + (M + 1) + ... + (M + k − 1) = k·M + k(k − 1)/2`.

**Đạt được cận trên.** Chiến lược "luôn chọn phần tử lớn nhất": thao tác `0` chọn `M`, thêm `M + 1`; vì `M + 1 > M >= mọi phần tử còn lại`, thao tác `1` chọn `M + 1`, thêm `M + 2`; ... cứ thế thao tác `t` chọn `M + t`. Vậy điểm bằng đúng cận trên.

Do cận trên đạt được, đó là **điểm lớn nhất**. Các phần tử khác `M` không bao giờ cần dùng.

(Ta cũng kiểm chứng bằng **vét cạn mọi cách chọn** ở mục 12.)

## 11. Phân tích Time / Space Complexity

`n = len(nums)`, `k` là số thao tác.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng bằng danh sách | `O(k · n)` | `O(n)` |
| 2. Mô phỏng bằng heap | `O(n + k log n)` | `O(n)` |
| 3. Một biến `cur` | `O(n + k)` | `O(1)` |
| 4. Công thức | `O(n)` | `O(1)` |
| 5. `sum(range(...))` | `O(n + k)` | `O(1)` |

Cận dưới thực tế là `O(n)` vì phải đọc `nums` để biết phần tử lớn nhất. Với `n, k <= 100` mọi cách đều tức thì.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: một phần tử; `k = 1` (đáp án là `max`); mọi phần tử bằng nhau; phần tử lớn nhất đứng đầu / giữa / cuối; số lớn nhất `100` với `k = 100`.

In [18]:
sols = [SolutionSimulate(), SolutionHeap(), SolutionLoop(), SolutionFormula(), SolutionRangeSum(), Solution()]
cases = [
    ("ví dụ 1", [1, 2, 3, 4, 5], 3, 18),
    ("ví dụ 2", [5, 5, 5], 2, 11),
    ("một phần tử, k=1", [7], 1, 7),
    ("một phần tử, k lớn", [1], 100, sum(range(1, 101))),
    ("k = 1 chọn max", [3, 9, 2], 1, 9),
    ("max ở đầu", [9, 1, 1], 3, 9 + 10 + 11),
    ("max ở cuối", [1, 1, 9], 3, 9 + 10 + 11),
    ("mọi phần tử bằng nhau", [4, 4, 4, 4], 4, 4 + 5 + 6 + 7),
    ("tối đa", [100] * 100, 100, 100 * 100 + 100 * 99 // 2),
    ("[3,9,2], k=4", [3, 9, 2], 4, 42),
]
for name, nums, k, exp in cases:
    for s in sols:
        got = s.maximizeSum(nums[:], k)
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 18
PASS ví dụ 2 -> 11
PASS một phần tử, k=1 -> 7
PASS một phần tử, k lớn -> 5050
PASS k = 1 chọn max -> 9
PASS max ở đầu -> 30
PASS max ở cuối -> 30
PASS mọi phần tử bằng nhau -> 22
PASS tối đa -> 14950
PASS [3,9,2], k=4 -> 42


### 12.2. Vét cạn mọi cách chọn trên dữ liệu nhỏ (chứng minh tham lam đúng)

Ta viết oracle **không dùng tham lam**: thử **mọi** cách chọn phần tử ở mỗi bước bằng đệ quy rồi lấy điểm lớn nhất. Nếu tham lam sai, oracle sẽ lộ ra.

In [19]:
import random
random.seed(2656)

def brute_all_choices(nums, k):
    if k == 0:
        return 0
    best = 0
    for i in range(len(nums)):
        m = nums[i]
        rest = nums[:i] + nums[i + 1:] + [m + 1]
        best = max(best, m + brute_all_choices(rest, k - 1))
    return best

for _ in range(300):
    n = random.randint(1, 4)
    k = random.randint(1, 4)
    nums = [random.randint(1, 6) for _ in range(n)]
    exp = brute_all_choices(nums, k)
    for s in sols:
        got = s.maximizeSum(nums[:], k)
        assert got == exp, (type(s).__name__, nums, k, got, exp)
print("300 ca nhỏ: mọi cách khớp vét cạn mọi lựa chọn")

300 ca nhỏ: mọi cách khớp vét cạn mọi lựa chọn


### 12.3. Đối chiếu ngẫu nhiên trên miền của đề

Sinh `nums` với `1 <= n <= 100`, giá trị `1..100`, `k` trong `1..100`; so các cách với mô phỏng danh sách (Cách 1) và tính chất: đáp án luôn `>= k · max`.

In [20]:
random.seed(7)
for _ in range(400):
    n = random.randint(1, 100)
    k = random.randint(1, 100)
    nums = [random.randint(1, 100) for _ in range(n)]
    exp = SolutionSimulate().maximizeSum(nums[:], k)
    for s in sols:
        assert s.maximizeSum(nums[:], k) == exp, (type(s).__name__,)
    assert exp >= k * max(nums)
    snapshot = nums[:]
    Solution().maximizeSum(nums, k)
    assert nums == snapshot            # không làm hỏng dữ liệu vào
print("400 ca ngẫu nhiên miền đề: OK, nums không bị sửa")

400 ca ngẫu nhiên miền đề: OK, nums không bị sửa


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chọn phần tử nhỏ nhất (hoặc theo thứ tự mảng)

In [21]:
from typing import List

def sai_chon_nho_nhat(nums: List[int], k: int) -> int:
    a = nums[:]
    score = 0
    for _ in range(k):
        m = min(a)
        a.remove(m)
        a.append(m + 1)
        score += m
    return score

print("sai :", sai_chon_nho_nhat([1, 2, 3, 4, 5], 3))     # 6
print("đúng:", Solution().maximizeSum([1, 2, 3, 4, 5], 3))   # 18

sai : 5
đúng: 18


**Vì sao sai:** đề muốn **tối đa** điểm, mỗi lần cộng đúng giá trị `m` được chọn ⇒ nên chọn phần tử lớn nhất. Chọn `min` cho `1 + 2 + 3 = 6`.

### Lỗi 2: Cộng `k` lần `max` mà quên phần tử mới `m + 1`

In [22]:
def sai_quen_tang(nums: List[int], k: int) -> int:
    return k * max(nums)

print("sai :", sai_quen_tang([1, 2, 3, 4, 5], 3))     # 15
print("đúng:", Solution().maximizeSum([1, 2, 3, 4, 5], 3))   # 18

sai : 15
đúng: 18


**Vì sao sai:** sau mỗi lần chọn `m`, phần tử `m + 1` xuất hiện và **lớn hơn** `m`, nên lần sau điểm cao hơn. Phần bù là `0 + 1 + ... + (k − 1) = k(k − 1)/2`.

### Lỗi 3: Dùng sai công thức `k(k + 1)/2` thay vì `k(k − 1)/2`

In [23]:
def sai_cong_thuc(nums: List[int], k: int) -> int:
    return k * max(nums) + k * (k + 1) // 2

print("sai :", sai_cong_thuc([5, 5, 5], 2))            # 13
print("đúng:", Solution().maximizeSum([5, 5, 5], 2))   # 11

sai : 13
đúng: 11


**Vì sao sai:** các số hạng là `M + 0, M + 1, ..., M + (k − 1)`, phần cộng thêm là `0 + 1 + ... + (k − 1) = k(k − 1)/2`. Công thức `k(k + 1)/2` sẽ tính `1 + 2 + ... + k` — lệch `k` đơn vị. Kiểm tra nhanh với `k = 1`: phần cộng thêm phải là `0`.

### Lỗi 4: Sửa mảng gốc và dùng lại (tác dụng phụ)

In [24]:
def sai_sua_mang_goc(nums: List[int], k: int) -> int:
    score = 0
    for _ in range(k):
        m = max(nums)
        nums.remove(m)          # sửa trực tiếp nums của người gọi
        nums.append(m + 1)
        score += m
    return score

data = [1, 2, 3, 4, 5]
print(sai_sua_mang_goc(data, 3), "| dữ liệu sau khi gọi:", data)    # data đã bị đổi
data2 = [1, 2, 3, 4, 5]
print(Solution().maximizeSum(data2, 3), "| dữ liệu sau khi gọi:", data2)

18 | dữ liệu sau khi gọi: [1, 2, 3, 4, 8]
18 | dữ liệu sau khi gọi: [1, 2, 3, 4, 5]


**Vì sao nên tránh:** nếu bạn gọi hàm hai lần trên cùng `nums`, lần hai thấy dữ liệu đã đổi và cho đáp án khác. Dùng bản sao (`nums[:]`) hoặc không sửa (công thức) cho an toàn. Ở LeetCode thường chạy lại với dữ liệu mới nên không lỗi, nhưng đó là thói quen xấu.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "thực hiện đúng `k` lần, mỗi lần chọn một phần tử, tối đa điểm" ⇒ nghĩ ngay **tham lam: chọn phần tử lớn nhất**.
- Khi một thao tác "thay phần tử bằng phần tử to hơn" ⇒ phần tử vừa thay thường lại là lớn nhất ⇒ chỉ cần theo dõi **một biến**.
- Nếu phải lấy lớn nhất nhiều lần nhưng thao tác phức tạp (không đoán được quy luật) ⇒ **max-heap** (`heapq` với số âm).
- Thấy tổng các số liên tiếp ⇒ nhớ công thức `k·M + k(k − 1)/2` hoặc `sum(range(M, M + k))`.
- Để chắc tham lam đúng, viết **vét cạn** trên dữ liệu nhỏ và so sánh (như mục 12.2).

## 15. Biến thể của bài

1. **Mỗi thao tác tăng thêm `d`** (thay vì `1`): các số hạng `M, M + d, ..., M + (k − 1)d` ⇒ `k·M + d·k(k − 1)/2`.
2. **Chọn phần tử nhỏ nhất để tối thiểu hoá điểm:** đối xứng, dùng min-heap.
3. **Maximize Sum Of Array After K Negations (LeetCode 1005):** tham lam khác: đổi dấu phần tử nhỏ nhất.
4. **Last Stone Weight (LeetCode 1046):** lặp "lấy lớn nhất, xử lý, thêm lại" bằng max-heap.
5. **Kth Largest Element in an Array (LeetCode 215):** dùng heap cho phần tử lớn thứ `k`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng danh sách | Làm đúng như đề | `O(k · n)` | `O(n)` | Hiểu bài, kiểm chứng |
| 2. Mô phỏng heap | Max-heap cho "lớn nhất" | `O(n + k log n)` | `O(n)` | Khi quy luật không đơn giản |
| 3. Một biến `cur` | Phần tử mới luôn lớn nhất | `O(n + k)` | `O(1)` | Rõ ràng, không công thức |
| 4. Công thức | `k·M + k(k − 1)/2` | `O(n)` | `O(1)` | **Nên nộp**, ngắn nhất |
| 5. `sum(range(...))` | Tổng k số liên tiếp | `O(n + k)` | `O(1)` | Dễ đọc nhất |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `sum(range(5, 8))`, `3 * 5 + 3 * 2 // 2`, `max([3, 9, 2])`, `[1, 2, 2].remove(2)` sửa mảng thành gì.

**Bài 2 (tính tay).** Với `nums = [3, 9, 2]`, `k = 4`: viết mảng sau từng lần chọn và điểm cuối.

**Bài 3 (điền TODO).** Hoàn thành `maximize_sum_formula_todo(nums, k)` bằng công thức.

In [25]:
def maximize_sum_formula_todo(nums, k):
    # TODO 1: M = phần tử lớn nhất của nums
    # TODO 2: return k * M + k * (k - 1) // 2
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `maximize_sum_heap(nums, k)` bằng `heapq` (max-heap với số âm) mà không nhìn Cách 2. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `maximize_sum_step(nums, k, d)`: mỗi thao tác thêm `m + d` (thay vì `m + 1`). (b) Viết `final_array(nums, k)` trả về **mảng cuối cùng** (đã sắp xếp tăng dần) sau `k` thao tác tham lam.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `range(5, 8)` là `5, 6, 7` ⇒ `18`; `3·5 + 3 = 18`; `max = 9`; `remove` trả `None` và chỉ xoá **một** số `2` đầu tiên. Bài 2: bắt đầu từ `9`, lần lượt `9, 10, 11, 12`.
- **Mức 2 (vừa):** Bài 3: `M = max(nums)` rồi `return k * M + k * (k - 1) // 2`. Bài 4: `heap = [-x for x in nums]`, `heapq.heapify`, rồi `m = -heap[0]`, `heapq.heapreplace(heap, -(m + 1))`.
- **Mức 3 (sâu):** Bài 5(a): các số hạng là `M, M + d, ..., M + (k − 1)d` nên đáp án `k·M + d·k(k − 1)/2` (vẫn chọn lớn nhất mỗi lần vì `m + d > m` khi `d > 0`). Bài 5(b): chạy mô phỏng đúng `k` lần rồi `sorted`; kết quả là mảng gốc bỏ `max` thêm `M + k`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
import heapq

# Bài 1
assert sum(range(5, 8)) == 18
assert 3 * 5 + 3 * 2 // 2 == 18
assert max([3, 9, 2]) == 9
lst = [1, 2, 2]
assert lst.remove(2) is None and lst == [1, 2]

# Bài 2
a, score = [3, 9, 2], 0
for _ in range(4):
    m = max(a)
    a.remove(m)
    a.append(m + 1)
    score += m
assert score == 9 + 10 + 11 + 12 == 42
assert sorted(a) == [2, 3, 13]

# Bài 3
def maximize_sum_formula_todo_ans(nums, k):
    M = max(nums)
    return k * M + k * (k - 1) // 2

assert maximize_sum_formula_todo_ans([1, 2, 3, 4, 5], 3) == 18
assert maximize_sum_formula_todo_ans([5, 5, 5], 2) == 11

# Bài 4
def maximize_sum_heap(nums, k):
    heap = [-x for x in nums]
    heapq.heapify(heap)
    score = 0
    for _ in range(k):
        m = -heap[0]
        score += m
        heapq.heapreplace(heap, -(m + 1))
    return score

assert maximize_sum_heap([1, 2, 3, 4, 5], 3) == 18
assert maximize_sum_heap([5, 5, 5], 2) == 11

# Bài 5
def maximize_sum_step(nums, k, d):
    M = max(nums)
    return k * M + d * k * (k - 1) // 2

assert maximize_sum_step([1, 2, 3, 4, 5], 3, 1) == 18
assert maximize_sum_step([5, 5, 5], 2, 3) == 5 + 8 == 13

def final_array(nums, k):
    a = nums[:]
    for _ in range(k):
        m = max(a)
        a.remove(m)
        a.append(m + 1)
    return sorted(a)

assert final_array([1, 2, 3, 4, 5], 3) == [1, 2, 3, 4, 8]
assert final_array([5, 5, 5], 2) == [5, 5, 7]
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tham lam: luôn chọn lớn nhất, công thức cấp số cộng)

```python
from typing import List

class Solution:
    def maximizeSum(self, nums: List[int], k: int) -> int:
        M = max(nums)
        return k * M + k * (k - 1) // 2      # M + (M+1) + ... + (M+k-1)
```

**Checklist**

- [ ] Chọn phần tử **lớn nhất** (không phải nhỏ nhất)?
- [ ] Biết rằng phần tử mới `m + 1` lại là lớn nhất ⇒ các giá trị lấy ra là `M, M+1, ..., M+k−1`?
- [ ] Công thức `k·M + k(k−1)/2` (không phải `k(k+1)/2`); thử `k = 1` ra `M`?
- [ ] Dùng `//` để kết quả là `int`?
- [ ] Không sửa `nums` gốc nếu không cần?
- [ ] Đã thử `k = 1`, một phần tử, mọi phần tử bằng nhau, `k = 100`?
- [ ] Signature đúng `maximizeSum(self, nums: List[int], k: int) -> int` và có `from typing import List`?